# 06 — R6: the approval threshold, and what a constraint costs per approved application

**ADIL** · MAIB AI 217 (AI in Finance) · SP Jain School of Global Management, Dubai · Krishna Mathur

This notebook closes the research question. Everything before it produced probabilities; a
probability is not a decision, and the gap between them is a cost matrix.

Four things happen here.

1. A cost-minimising cutoff for each rung, from `spine.decisions`.
2. The gap between that cutoff and the one arithmetic predicts — **that gap is a measurement
   of miscalibration**, and it is why calibration was a precondition rather than a rung.
3. **R6**: one cutoff for everyone, constrained so approval-rate disparity stays above the
   four-fifths line. Per-group cutoffs were shown in notebook 05 to be disparate treatment,
   so the constraint moves the single threshold and the cost of moving it is the finding.
4. The headline table.

Two disciplines carried through from `adil.costs`. The cost **ratio** is not invented — it is
the one published ratio available anywhere in this project's data — and the whole curve is
reported around it so no single assumed figure drives the conclusion. And **no primary result
is in dirhams**: Home Credit's currency is anonymised, so dirham figures are a labelled
scenario and are marked as such at every appearance.

Outputs `reports/decision_table.md`, `metrics/r6.json` and `metrics/headline.json`.

## 1. The cost model

Home Credit publishes no cost of default and no margin. Any specific figure would be a number
typed into a notebook and then reported as though it meant something.

The one published ratio in this project's data is UCI German Credit's documented cost matrix.
It is a ratio from a different country in a different currency, so only the **ratio** transfers
— but a ratio is all a threshold needs, because scaling both costs equally moves the total and
leaves the cutoff exactly where it was.

In [ ]:
import json
import warnings

import numpy as np
import pandas as pd
from spine.decisions import expected_cost_curve, optimal_threshold
from spine.fairness import disparity_ratio, group_metrics

from adil import costs, paths
from adil import split as sp

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)

processed = paths.processed_dir()
frame = pd.read_parquet(processed / "adil_frame.parquet")
splits = pd.read_parquet(processed / "split_index.parquet")["split"].values
r0_predictions = pd.read_parquet(processed / "r0_predictions.parquet")
challenger_predictions = pd.read_parquet(processed / "challenger_predictions.parquet")
is_test = splits == "test"

metrics_dir = paths.metrics_dir()
r0_metrics = json.loads((metrics_dir / "r0.json").read_text())
r5_metrics = json.loads((metrics_dir / "r5.json").read_text())
rung_metrics = {
    name: json.loads((metrics_dir / f"{name.lower()}.json").read_text())
    for name in ("R1", "R3", "R4")
}

probability = {
    "R0": r0_predictions.loc[is_test, "prob_calibrated"].values,
    "R1": challenger_predictions.loc[is_test, "R1_prob_calibrated"].values,
    "R3": challenger_predictions.loc[is_test, "R3_prob_calibrated"].values,
    "R4": challenger_predictions.loc[is_test, "R4_prob_calibrated"].values,
}
default = frame.loc[is_test, "TARGET"].values
exposure = frame.loc[is_test, "AMT_CREDIT"].values

print(costs.GERMAN_CREDIT_SOURCE)
print("")
reference_matrix = costs.cost_matrix()
print(
    f"reference cost matrix (ratio {costs.GERMAN_CREDIT_COST_RATIO:g}:1), "
    f"indexed [true][predicted]:"
)
print(reference_matrix)

## 2. The cost-minimising cutoff, and what its distance from arithmetic means

`spine.decisions.optimal_threshold` searches the observed probabilities rather than assuming
the model is calibrated. For a perfectly calibrated model that search converges on the
closed-form answer, `co / (co + cu)`.

Where the two disagree, **the disagreement measures miscalibration, and the analytic value is
the one that is wrong** — it assumes a calibration the model may not have. This is the concrete
reason calibration was treated as a precondition in notebooks 02 and 03 rather than as a rung
of the ladder: a model that ranks well but is miscalibrated cannot support a cost-based cutoff
at all, however good its AUC.

In [ ]:
analytic = costs.analytic_threshold(reference_matrix)

threshold_rows = []
for rung, scores in probability.items():
    empirical, expected = optimal_threshold(default, scores, reference_matrix)
    accounting = costs.decision_costs(default, scores, empirical, reference_matrix)
    threshold_rows.append(
        {
            "rung": rung,
            "empirical threshold": empirical,
            "analytic threshold": analytic,
            "gap": empirical - analytic,
            "expected cost per application": expected,
            "approval rate": accounting["approval_rate"],
            "bad rate among approved": accounting["bad_rate_among_approved"],
        }
    )

thresholds = pd.DataFrame(threshold_rows).set_index("rung")
print(f"analytic threshold for a {costs.GERMAN_CREDIT_COST_RATIO:g}:1 ratio: {analytic:.6f}")
thresholds.round(5)

In [ ]:
curve = expected_cost_curve(default, probability["R4"], reference_matrix)
best = curve.loc[curve["expected_cost"].idxmin()]
print(f"R4 expected-cost curve: {len(curve):,} candidate thresholds")
print(
    f"minimum at threshold {best['threshold']:.5f}, "
    f"expected cost {best['expected_cost']:.5f} per application"
)
curve.iloc[:: max(1, len(curve) // 12)].round(5)

## 3. Sensitivity to the cost ratio

The reference ratio is published rather than invented, but it still comes from a different
book in a different country. A conclusion that only holds at 5:1 is not a conclusion, so every
quantity is reported across the grid.

In [ ]:
sensitivity_rows = []
for ratio in costs.RATIO_GRID:
    matrix = costs.cost_matrix(ratio)
    for rung, scores in probability.items():
        empirical, _ = optimal_threshold(default, scores, matrix)
        accounting = costs.decision_costs(default, scores, empirical, matrix)
        sensitivity_rows.append(
            {
                "ratio": ratio,
                "rung": rung,
                "threshold": empirical,
                "approval rate": accounting["approval_rate"],
                "cost per approved": accounting["cost_per_approved"],
            }
        )

sensitivity = pd.DataFrame(sensitivity_rows)
print("cost per approved application, in units of the margin forgone by declining a")
print("good applicant. Lower is better; the column is comparable within a ratio, not across.")
sensitivity.pivot_table(index="ratio", columns="rung", values="cost per approved").round(4)

In [ ]:
print("approval rate at the cost-optimal cutoff")
sensitivity.pivot_table(index="ratio", columns="rung", values="approval rate").round(4)

## 4. R6 — a fairness band on the single cutoff

Notebook 05 established the constraint this rung has to respect. Equalising error rates
required a **different cutoff for each sex**, which is disparate treatment, and it did not even
deliver equalised odds. So R6 does not adjust per group. It keeps **one cutoff for everyone**
and moves that cutoff until approval-rate disparity clears the four-fifths line on both sex and
age band.

That line is a US employment-screening heuristic rather than a legal standard anywhere this
project applies. It is used here as a declared, checkable bound — a stand-in for the kind of
threshold a credit policy committee would set — not as a claim about what any regulator
requires.

R6's model is R4. Only the cutoff changes, so discrimination metrics are identical by
construction and the entire cost of the rung appears in the money and the approval rate.

In [ ]:
DISPARITY_FLOOR = 0.80

sex = frame.loc[is_test, "CODE_GENDER"].astype(str).values
usable_sex = sex != "XNA"
age_band = pd.cut(
    (-frame.loc[is_test, "DAYS_BIRTH"] / 365.25).values,
    [0, 25, 35, 45, 55, 200],
    right=False,
    labels=["<25", "25-34", "35-44", "45-54", "55+"],
).astype(str)
repaid = 1 - default


def approval_disparity(scores, threshold):
    approved = (scores < threshold).astype(int)
    sex_rates = group_metrics(repaid[usable_sex], approved[usable_sex], sex[usable_sex])
    age_rates = group_metrics(repaid, approved, age_band)
    return (
        disparity_ratio(sex_rates, "selection_rate"),
        disparity_ratio(age_rates, "selection_rate"),
    )


scores_r4 = probability["R4"]
candidates = np.unique(np.quantile(scores_r4, np.linspace(0.001, 0.999, 400)))
band_rows = []
for threshold in candidates:
    accounting = costs.decision_costs(default, scores_r4, threshold, reference_matrix)
    if accounting["n_approved"] == 0:
        continue
    sex_disparity, age_disparity = approval_disparity(scores_r4, threshold)
    band_rows.append(
        {
            "threshold": float(threshold),
            "approval rate": accounting["approval_rate"],
            "cost per approved": accounting["cost_per_approved"],
            "cost per application": accounting["cost_per_application"],
            "sex disparity": sex_disparity,
            "age disparity": age_disparity,
            "feasible": bool(sex_disparity >= DISPARITY_FLOOR and age_disparity >= DISPARITY_FLOOR),
        }
    )

band = pd.DataFrame(band_rows)
print(f"searched {len(band)} candidate cutoffs for R4")
print(
    f"clearing a {DISPARITY_FLOOR:.2f} floor on both sex and age band: "
    f"{int(band['feasible'].sum())}"
)
print(
    f"sex disparity range   : {band['sex disparity'].min():.3f} to "
    f"{band['sex disparity'].max():.3f}"
)
print(
    f"age disparity range   : {band['age disparity'].min():.3f} to "
    f"{band['age disparity'].max():.3f}"
)

In [ ]:
unconstrained_threshold = float(thresholds.loc["R4", "empirical threshold"])
unconstrained = costs.decision_costs(default, scores_r4, unconstrained_threshold, reference_matrix)
unconstrained_sex, unconstrained_age = approval_disparity(scores_r4, unconstrained_threshold)

feasible = band[band["feasible"]]
if len(feasible):
    chosen = feasible.loc[feasible["cost per application"].idxmin()]
    r6_threshold = float(chosen["threshold"])
    r6_status = "constraint satisfied"
else:
    # No cutoff clears the floor. Report the best available rather than inventing one.
    chosen = band.loc[band[["sex disparity", "age disparity"]].min(axis=1).idxmax()]
    r6_threshold = float(chosen["threshold"])
    r6_status = "infeasible — no single cutoff clears the floor"

r6 = costs.decision_costs(default, scores_r4, r6_threshold, reference_matrix)
r6_sex, r6_age = approval_disparity(scores_r4, r6_threshold)

comparison = pd.DataFrame(
    [
        {
            "cutoff": "R4 cost-optimal",
            "threshold": unconstrained_threshold,
            "approval rate": unconstrained["approval_rate"],
            "sex disparity": unconstrained_sex,
            "age disparity": unconstrained_age,
            "cost per approved": unconstrained["cost_per_approved"],
            "cost per application": unconstrained["cost_per_application"],
        },
        {
            "cutoff": "R6 fairness-banded",
            "threshold": r6_threshold,
            "approval rate": r6["approval_rate"],
            "sex disparity": r6_sex,
            "age disparity": r6_age,
            "cost per approved": r6["cost_per_approved"],
            "cost per application": r6["cost_per_application"],
        },
    ]
).set_index("cutoff")
print(f"R6 status: {r6_status}")
comparison.round(5)

## 5. Money

Everything above is in units of the margin forgone by declining one good applicant, which is
the right unit for choosing a cutoff and a useless one for talking to a business.

Converting needs an exposure, and `AMT_CREDIT` is real data rather than an assumption. What is
assumed is the loss given default and the margin rate — both stated, and both chosen so their
**ratio equals the published German Credit ratio**, so the money scale and the reference point
are one assumption rather than two pulling apart.

Dirhams are a different matter. Home Credit's currency is anonymised and unscaled, so no
dirham figure is computable from this data. The AED column below is a rescaling of the dataset
result onto a stated mean exposure, and it carries its caveat wherever it appears.

In [ ]:
LGD = float(costs.AED_SCENARIO["assumed_loss_given_default"])
MARGIN_RATE = float(costs.AED_SCENARIO["assumed_lifetime_margin_rate"])
AED_MEAN_EXPOSURE = float(costs.AED_SCENARIO["assumed_mean_exposure_aed"])

mean_exposure = float(exposure.mean())
margin_unit = MARGIN_RATE * mean_exposure
aed_scale = AED_MEAN_EXPOSURE / mean_exposure

print(
    f"assumed LGD {LGD:.2f} / margin rate {MARGIN_RATE:.2f} = ratio "
    f"{LGD / MARGIN_RATE:g}:1, matching the published "
    f"{costs.GERMAN_CREDIT_COST_RATIO:g}:1"
)
print(f"mean exposure on test : {mean_exposure:,.0f} dataset currency units")
print(f"one margin unit       : {margin_unit:,.0f} dataset currency units")
print("")
print(costs.SCENARIO_CAVEAT)

In [ ]:
def in_money(cost_in_margin_units):
    dataset_units = cost_in_margin_units * margin_unit
    return dataset_units, dataset_units * aed_scale


money_rows = []
for label, accounting in [
    ("R4 cost-optimal", unconstrained),
    ("R6 fairness-banded", r6),
]:
    dataset_units, aed = in_money(accounting["cost_per_approved"])
    money_rows.append(
        {
            "cutoff": label,
            "approval rate": accounting["approval_rate"],
            "cost per approved (dataset units)": dataset_units,
            "cost per approved (AED, SCENARIO)": aed,
        }
    )
money = pd.DataFrame(money_rows).set_index("cutoff")
print("SCENARIO — assumed parameters, not a finding")
money.round(2)

### Exposure-weighted sensitivity

`spine.decisions.optimal_threshold` takes one cost matrix for the whole book, so every
applicant above is treated as carrying the same loss and the same margin. On this data they do
not: exposures span a factor of ninety.

A constant matrix is still the right primary result — it is the cutoff a policy committee sets
when it will not price the decision per applicant — but the assumption is load-bearing enough
to be checked rather than mentioned. Below, each applicant's own `AMT_CREDIT` sets their loss
and their margin, and the cutoff is re-optimised directly.

In [ ]:
loss_by_application = LGD * exposure
margin_by_application = MARGIN_RATE * exposure


def exposure_weighted_cost(threshold):
    approved = probability["R4"] < threshold
    approved_bad = approved & (default == 1)
    declined_good = ~approved & (default == 0)
    total = loss_by_application[approved_bad].sum() + margin_by_application[declined_good].sum()
    n_approved = int(approved.sum())
    return total, n_approved, (total / n_approved if n_approved else np.nan)


weighted = pd.DataFrame(
    [
        {"threshold": float(t), "cost per approved": exposure_weighted_cost(float(t))[2]}
        for t in candidates
    ]
).dropna()
weighted_best = weighted.loc[weighted["cost per approved"].idxmin()]

constant_matrix_cost = exposure_weighted_cost(unconstrained_threshold)[2]
print(f"constant-matrix cutoff        : {unconstrained_threshold:.5f}")
print(f"exposure-weighted cutoff      : {weighted_best['threshold']:.5f}")
print(
    f"shift in cutoff               : {weighted_best['threshold'] - unconstrained_threshold:+.5f}"
)
print("")
print(f"cost per approved at the constant-matrix cutoff : {constant_matrix_cost:,.0f}")
print(
    f"cost per approved at its own optimum            : {weighted_best['cost per approved']:,.0f}"
)
print(
    f"penalty for ignoring exposure                   : "
    f"{constant_matrix_cost - weighted_best['cost per approved']:,.0f} "
    f"({100 * (constant_matrix_cost / weighted_best['cost per approved'] - 1):.2f}%)"
)

## 6. The headline table

Rows are rungs of the ladder. R0 to R4 change the model. R5 changes nothing and returns a
verdict. R6 changes only the cutoff, so its discrimination metrics are R4's by construction and
its entire cost is in the money column.

Discrimination and calibration are on calibrated test predictions. Approval rate, disparity and
cost are at each rung's own cost-optimal cutoff under the reference ratio.

In [ ]:
flip = r5_metrics["verdicts"]
headline_rows = []
for rung in ("R0", "R1", "R3", "R4", "R6"):
    if rung == "R6":
        scores, accounting = scores_r4, r6
        threshold, features = r6_threshold, rung_metrics["R4"]["n_features"]
        sex_disparity, age_disparity = r6_sex, r6_age
        source = rung_metrics["R4"]["metrics_test_calibrated"]
        flip_rate = flip["R4"]["flip_rate"]
        verdict = flip["R4"]["verdict"]
    else:
        scores = probability[rung]
        threshold = float(thresholds.loc[rung, "empirical threshold"])
        accounting = costs.decision_costs(default, scores, threshold, reference_matrix)
        sex_disparity, age_disparity = approval_disparity(scores, threshold)
        if rung == "R0":
            source = r0_metrics["calibrated_test"]
            features = r0_metrics["n_characteristics"]
        else:
            source = rung_metrics[rung]["metrics_test_calibrated"]
            features = rung_metrics[rung]["n_features"]
        flip_rate = flip[rung]["flip_rate"]
        verdict = flip[rung]["verdict"]

    dataset_units, aed = in_money(accounting["cost_per_approved"])
    per_application, _ = in_money(accounting["cost_per_application"])
    headline_rows.append(
        {
            "rung": rung,
            "features": features,
            "PR-AUC": source["pr_auc"],
            "AUC": source["auc"],
            "KS": source["ks"],
            "Gini": source["gini"],
            "Brier": source["brier"],
            "ECE": source["ece"],
            "flip rate": flip_rate,
            "R5": verdict,
            "threshold": threshold,
            "approval rate": accounting["approval_rate"],
            "disparity sex": sex_disparity,
            "disparity age": age_disparity,
            "cost/approved (units)": accounting["cost_per_approved"],
            "cost/approved (dataset)": dataset_units,
            "cost/application (dataset)": per_application,
            "cost/approved (AED, SCENARIO)": aed,
        }
    )

headline = pd.DataFrame(headline_rows).set_index("rung")
headline.round(5)

In [ ]:
print("what each step of the ladder cost, per approved application")
steps = []
order = ["R0", "R1", "R3", "R4", "R6"]
labels = {
    "R1": "model class (R1 - R0)",
    "R3": "monotonicity (R3 - R1)",
    "R4": "feature budget (R4 - R3)",
    "R6": "fairness band (R6 - R4)",
}
for previous, current in zip(order[:-1], order[1:], strict=True):
    steps.append(
        {
            "step": labels[current],
            "PR-AUC": headline.loc[current, "PR-AUC"] - headline.loc[previous, "PR-AUC"],
            "approval rate": (
                headline.loc[current, "approval rate"] - headline.loc[previous, "approval rate"]
            ),
            "disparity age": (
                headline.loc[current, "disparity age"] - headline.loc[previous, "disparity age"]
            ),
            "cost/approved (dataset)": (
                headline.loc[current, "cost/approved (dataset)"]
                - headline.loc[previous, "cost/approved (dataset)"]
            ),
            "cost/application (dataset)": (
                headline.loc[current, "cost/application (dataset)"]
                - headline.loc[previous, "cost/application (dataset)"]
            ),
        }
    )
pd.DataFrame(steps).set_index("step").round(5)

### Two denominators, moving in opposite directions

R6 shows a **lower** cost per approved application than R4 and a **higher** cost per
application. Both are correct, and reporting only the first would be misleading, so both are
in the table.

The cutoff that minimises cost per *application* is not the cutoff that minimises cost per
*approved* application, because raising the threshold changes the numerator and the denominator
at different rates. Approving more people takes on more bad loans — the numerator rises — but
it also spreads that cost over a larger book. Over this range the denominator wins.

So the fairness band did not make lending cheaper. It raised total expected cost while
approving a larger, slightly worse book, and the per-approved figure fell because the book grew
faster than the losses did. A lender optimises total cost; the per-approved figure is a unit
economic, and the research question asked for it, but on its own it can be improved by simply
approving more people.

Every rung's cutoff in the headline table is its **cost-per-application** optimum, which is the
quantity a lender actually minimises. The per-approved column is then read off that cutoff
rather than separately optimised — otherwise the ladder would be comparing rungs at cutoffs
chosen by a metric no one uses to set policy.

In [ ]:
for label, accounting in [("R4 cost-optimal", unconstrained), ("R6 banded", r6)]:
    print(
        f"{label:>18}  approval {accounting['approval_rate']:.4f}  "
        f"cost/application {accounting['cost_per_application']:.5f}  "
        f"cost/approved {accounting['cost_per_approved']:.5f}"
    )
total_change = 100 * (r6["cost_per_application"] / unconstrained["cost_per_application"] - 1)
book_change = 100 * (r6["n_approved"] / unconstrained["n_approved"] - 1)
print("")
print(f"total expected cost per application: {total_change:+.2f}%")
print(f"size of the approved book          : {book_change:+.2f}%")
print("The band costs money. The per-approved figure falls because the book grew faster.")

## 7. Persist

In [ ]:
payload_r6 = {
    "rung": "R6",
    "kind": "threshold constraint — changes the cutoff, not the model",
    "model": "R4",
    "seed": sp.SEED,
    "disparity_floor": DISPARITY_FLOOR,
    "status": r6_status,
    "note": (
        "One cutoff for everyone. Notebook 05 showed that per-group cutoffs close the "
        "equalised-odds gap only partially and cost disparate treatment to do it, so this "
        "rung moves the single threshold instead and reports what that costs."
    ),
    "cost_ratio": costs.GERMAN_CREDIT_COST_RATIO,
    "cost_ratio_source": costs.GERMAN_CREDIT_SOURCE,
    "unconstrained": {
        "threshold": unconstrained_threshold,
        **{k: float(v) for k, v in unconstrained.items()},
        "sex_disparity": float(unconstrained_sex),
        "age_disparity": float(unconstrained_age),
    },
    "constrained": {
        "threshold": r6_threshold,
        **{k: float(v) for k, v in r6.items()},
        "sex_disparity": float(r6_sex),
        "age_disparity": float(r6_age),
    },
    "exposure_weighted_sensitivity": {
        "constant_matrix_threshold": unconstrained_threshold,
        "exposure_weighted_threshold": float(weighted_best["threshold"]),
        "cost_per_approved_at_constant_matrix_cutoff": float(constant_matrix_cost),
        "cost_per_approved_at_own_optimum": float(weighted_best["cost per approved"]),
        "exposure_max_over_min": float(exposure.max() / exposure.min()),
    },
    "scenario": costs.AED_SCENARIO,
}
(metrics_dir / "r6.json").write_text(json.dumps(payload_r6, indent=2, default=float) + "\n")

payload_headline = {
    "seed": sp.SEED,
    "split": "test",
    "cost_ratio": costs.GERMAN_CREDIT_COST_RATIO,
    "analytic_threshold": analytic,
    "money_basis": {
        "mean_exposure_dataset_units": mean_exposure,
        "margin_unit_dataset_units": margin_unit,
        "aed_rescaling_factor": aed_scale,
        "caveat": costs.SCENARIO_CAVEAT,
        **{k: v for k, v in costs.AED_SCENARIO.items() if k != "caveat"},
    },
    "thresholds": thresholds.reset_index().to_dict("records"),
    "ladder": headline.reset_index().to_dict("records"),
    "steps": steps,
}
(metrics_dir / "headline.json").write_text(
    json.dumps(payload_headline, indent=2, default=float) + "\n"
)
sensitivity.to_parquet(processed / "cost_ratio_sensitivity.parquet", index=False)
band.to_parquet(processed / "fairness_band_search.parquet", index=False)
print("wrote metrics/r6.json and metrics/headline.json")

In [ ]:
r0h, r1h, r4h, r6h = (headline.loc[r] for r in ("R0", "R1", "R4", "R6"))
total_change = 100 * (r6["cost_per_application"] / unconstrained["cost_per_application"] - 1)
book_change = 100 * (r6["n_approved"] / unconstrained["n_approved"] - 1)
gap_r4 = float(thresholds.loc["R4", "gap"])
worst_gap_rung = thresholds["gap"].abs().idxmax()
worst_gap = float(thresholds.loc[worst_gap_rung, "gap"])
exposure_penalty = float(constant_matrix_cost - weighted_best["cost per approved"])
exposure_penalty_pct = 100 * (constant_matrix_cost / weighted_best["cost per approved"] - 1)

lines = [
    "# ADIL — the decision table",
    "",
    "Generated by `notebooks/06_threshold.ipynb`. Every number is computed, not typed.",
    "",
    "MAIB AI 217 · SP Jain School of Global Management, Dubai · Krishna Mathur",
    "",
    "## The cost model",
    "",
    "Home Credit publishes no cost of default and no margin, so any specific figure would be",
    "a number typed into a notebook and reported as though it meant something. The one",
    "published ratio available anywhere in this project's data is used instead:",
    "",
    f"> {costs.GERMAN_CREDIT_SOURCE}",
    "",
    f"Reference ratio **{costs.GERMAN_CREDIT_COST_RATIO:g}:1**. Only a ratio can move a",
    "threshold — scaling both costs equally moves the total and leaves the cutoff where it",
    "was — so a ratio from another book still transfers even though its amounts do not.",
    "Every result is reported across a grid from 1:1 to 20:1 so no single figure drives it.",
    "",
    "## Miscalibration, measured by the threshold gap",
    "",
    "`spine.decisions.optimal_threshold` searches the observed probabilities instead of",
    "assuming calibration. For a perfectly calibrated model that search converges on the",
    f"closed form, which at {costs.GERMAN_CREDIT_COST_RATIO:g}:1 is **{analytic:.5f}**.",
    "Where the two disagree the disagreement measures miscalibration, and the analytic value",
    "is the wrong one — it assumes a property the model may not have.",
    "",
    "| Rung | Empirical cutoff | Analytic | Gap | Approval rate | Bad rate among approved |",
    "|---|---:|---:|---:|---:|---:|",
]
for rung, row in thresholds.iterrows():
    lines.append(
        f"| {rung} | {row['empirical threshold']:.5f} | {row['analytic threshold']:.5f} | "
        f"{row['gap']:+.5f} | {row['approval rate']:.4f} | "
        f"{row['bad rate among approved']:.4f} |"
    )
lines += [
    "",
    f"The largest gap is {worst_gap:+.5f} at {worst_gap_rung}. This is the concrete reason",
    "calibration was treated as a precondition in notebooks 02 and 03 rather than as a rung:",
    "a model that ranks well but is miscalibrated cannot support a cost-based cutoff at all,",
    "however good its AUC.",
    "",
    "## R6 — a fairness band on one cutoff",
    "",
    "Notebook 05 established the constraint. Equalising error rates required a different",
    "cutoff per sex, which is disparate treatment, and it closed the TPR gap without",
    "delivering equalised odds. R6 therefore keeps **one cutoff for everyone** and moves that",
    f"cutoff until approval-rate disparity clears **{DISPARITY_FLOOR:.2f}** on both sex and",
    "age band.",
    "",
    "That line is a US employment-screening heuristic, not a legal standard anywhere this",
    "project applies. It stands in for the kind of bound a credit policy committee would set,",
    "and it is declared and checkable rather than a claim about any regulator's requirement.",
    "",
    f"Status: **{r6_status}**.",
    "",
    "| Cutoff | Threshold | Approval rate | Sex disparity | Age disparity | "
    "Cost/application | Cost/approved |",
    "|---|---:|---:|---:|---:|---:|---:|",
]
for label, row in comparison.iterrows():
    lines.append(
        f"| {label} | {row['threshold']:.5f} | {row['approval rate']:.4f} | "
        f"{row['sex disparity']:.4f} | {row['age disparity']:.4f} | "
        f"{row['cost per application']:.5f} | {row['cost per approved']:.5f} |"
    )
lines += [
    "",
    "R6's model is R4. Only the cutoff moves, so every discrimination metric is identical by",
    "construction and the whole cost of the rung shows up in the approval rate and the money.",
    "",
    "## The headline table",
    "",
    "Discrimination and calibration on calibrated test predictions. Approval rate, disparity",
    "and cost at each rung's own cost-optimal cutoff under the reference ratio. PR-AUC leads:",
    "at an 8% base rate AUC flatters.",
    "",
    "| Rung | Feat. | PR-AUC | AUC | KS | Gini | Brier | ECE | Flip | R5 | Appr. | "
    "Disp. sex | Disp. age | Cost/appl. | Cost/appr. |",
    "|---|---:|---:|---:|---:|---:|---:|---:|---:|---|---:|---:|---:|---:|---:|",
]
for rung, row in headline.iterrows():
    lines.append(
        f"| {rung} | {int(row['features'])} | {row['PR-AUC']:.4f} | {row['AUC']:.4f} | "
        f"{row['KS']:.4f} | {row['Gini']:.4f} | {row['Brier']:.5f} | {row['ECE']:.5f} | "
        f"{row['flip rate']:.3f} | {row['R5']} | {row['approval rate']:.4f} | "
        f"{row['disparity sex']:.4f} | {row['disparity age']:.4f} | "
        f"{row['cost/application (dataset)']:,.0f} | "
        f"{row['cost/approved (dataset)']:,.0f} |"
    )
lines += [
    "",
    "Both cost columns are in dataset currency units. `Cost/appl.` is total expected cost",
    "divided by every application seen; `Cost/appr.` divides by the approved book only.",
    "",
    "### What each step cost",
    "",
    "| Step | PR-AUC | Approval rate | Age disparity | Cost/application | Cost/approved |",
    "|---|---:|---:|---:|---:|---:|",
]
for step in steps:
    lines.append(
        f"| {step['step']} | {step['PR-AUC']:+.4f} | {step['approval rate']:+.4f} | "
        f"{step['disparity age']:+.4f} | {step['cost/application (dataset)']:+,.0f} | "
        f"{step['cost/approved (dataset)']:+,.0f} |"
    )
lines += [
    "",
    "### The two denominators move in opposite directions",
    "",
    "R6 shows a **lower** cost per approved application and a **higher** cost per",
    "application. Both are correct, and reporting only the first would be misleading.",
    "",
    "The cutoff minimising cost per *application* is not the cutoff minimising cost per",
    "*approved* application: raising the threshold changes numerator and denominator at",
    "different rates. Approving more people takes on more bad loans, but spreads the cost",
    "over a larger book, and over this range the denominator wins.",
    "",
    "So the fairness band did not make lending cheaper. It raised total expected cost by",
    f"**{total_change:+.2f}%** per application while growing the approved book by",
    f"**{book_change:+.2f}%**, and the per-approved figure fell because the book grew faster",
    "than the losses did. A lender minimises total cost; the per-approved figure is a unit",
    "economic, and the research question asked for it, but on its own it can be improved by",
    "simply approving more people.",
    "",
    "Every cutoff in the headline table is its rung's **cost-per-application** optimum, which",
    "is what a lender actually minimises. The per-approved column is read off that cutoff",
    "rather than separately optimised — otherwise the ladder would compare rungs at cutoffs",
    "chosen by a metric nobody sets policy with.",
]
lines += [
    "",
    "## Money",
    "",
    "Costs above are in units of the margin forgone by declining one good applicant — the",
    "right unit for choosing a cutoff, a useless one for talking to a business.",
    "",
    "Converting requires an exposure. `AMT_CREDIT` is real data. The loss given default",
    f"({LGD:.2f}) and the margin rate ({MARGIN_RATE:.2f}) are assumptions, stated here and",
    "chosen so their ratio equals the published German Credit ratio — so the money scale and",
    "the reference point are one assumption rather than two pulling apart.",
    "",
    f"- Mean exposure on the test split: **{mean_exposure:,.0f}** dataset currency units",
    f"- One margin unit: **{margin_unit:,.0f}** dataset currency units",
    "",
    "### AED scenario",
    "",
    f"> {costs.SCENARIO_CAVEAT}",
    "",
    f"> {costs.AED_SCENARIO['basis']}",
    "",
    "| Cutoff | Approval rate | Cost/approved (dataset units) | Cost/approved (AED, SCENARIO) |",
    "|---|---:|---:|---:|",
]
for label, row in money.iterrows():
    lines.append(
        f"| {label} | {row['approval rate']:.4f} | "
        f"{row['cost per approved (dataset units)']:,.0f} | "
        f"{row['cost per approved (AED, SCENARIO)']:,.0f} |"
    )
lines += [
    "",
    "### Exposure-weighted sensitivity",
    "",
    "`spine.decisions.optimal_threshold` takes one cost matrix for the whole book, so every",
    "applicant above carries the same assumed loss and margin. On this data they do not:",
    f"exposures span a factor of **{exposure.max() / exposure.min():.0f}**.",
    "",
    "A constant matrix remains the right primary result — it is the cutoff a policy committee",
    "sets when it will not price each decision individually — but the assumption is",
    "load-bearing enough to check rather than mention. Giving each applicant their own loss",
    "and margin from their own `AMT_CREDIT`:",
    "",
    f"- Constant-matrix cutoff: **{unconstrained_threshold:.5f}**",
    f"- Exposure-weighted cutoff: **{float(weighted_best['threshold']):.5f}**",
    f"- Penalty for ignoring exposure: **{exposure_penalty:,.0f}** dataset currency units per",
    f"  approved application ({exposure_penalty_pct:.2f}%)",
    "",
    "## Limitations",
    "",
    "- The cost **ratio** is published; the loss given default, the margin rate and the AED",
    "  exposure are assumed. No dirham figure here is a measurement and none should be quoted",
    "  as one.",
    "- The reference ratio comes from a German consumer book of the 1990s. It transfers as a",
    "  ratio and in no other way.",
    "- Costs are single-period: no recovery timing, no prepayment, no cross-sell, and no",
    "  second-order effect of a declined applicant taking their business elsewhere.",
    "- The four-fifths floor is a screening heuristic used here as a declared bound, not a",
    "  legal standard in this jurisdiction.",
    "- Disparity and cost are point estimates on one test split. Notebook 03's intervals cover",
    "  discrimination metrics only.",
    "- Public competition data, not UAE consumer data; no figure transfers to an Emirati book.",
    "",
]
path = paths.reports_dir() / "decision_table.md"
path.write_text("\n".join(lines) + "\n")
print(f"wrote {path} ({len(lines)} lines)")